# Methylation Probe SVA Demo

This notebook is a split-out module from the larger SVA demonstration project.

It includes:
- shared setup and helper functions
- a focused synthetic use case
- commented R code for teaching and adaptation


# Surrogate Variable Analysis (SVA) in Bioinformatics

This notebook is a **large, self-contained R notebook** that demonstrates:

- what surrogate variables are
- why hidden confounding matters in omics studies
- how SVA can be simulated and applied
- how it compares to naive analysis
- how it can be used in several synthetic but realistic bioinformatics use cases

Included use cases:
1. Bulk RNA-seq differential expression with hidden batch effects
2. Microarray-style gene expression with latent technical variation
3. DNA methylation probe analysis with hidden chip/position effects
4. Multi-cohort integration example
5. Case-control biomarker discovery with nuisance structure
6. Survival-associated expression signatures with latent confounding
7. eQTL-like setting with hidden sample structure
8. Pathway-level interpretation after SVA correction

The code is intentionally long and heavily commented so it can serve as a teaching demo.


## Notes

- The datasets are synthetic and made up for demonstration.
- The code is written in **R** and stored in notebook code cells.
- The notebook does not depend on any proprietary data.
- Some sections use package checks and fallbacks so the notebook remains readable even if not all packages are installed.


In [ ]:
# ============================================================
# SECTION 1: GLOBAL SETUP
# ============================================================

options(stringsAsFactors = FALSE)
set.seed(12345)

cat("Starting SVA demonstration notebook...\n")

required_pkgs <- c(
  "stats",
  "utils",
  "graphics",
  "grDevices",
  "methods"
)

optional_pkgs <- c(
  "sva",
  "limma",
  "ggplot2",
  "matrixStats",
  "survival"
)

check_package <- function(pkg) {
  is_installed <- requireNamespace(pkg, quietly = TRUE)
  message(sprintf("Package %-12s installed: %s", pkg, is_installed))
  return(is_installed)
}

pkg_status <- lapply(c(required_pkgs, optional_pkgs), check_package)
names(pkg_status) <- c(required_pkgs, optional_pkgs)

have_sva <- isTRUE(pkg_status[["sva"]])
have_limma <- isTRUE(pkg_status[["limma"]])
have_ggplot2 <- isTRUE(pkg_status[["ggplot2"]])
have_matrixStats <- isTRUE(pkg_status[["matrixStats"]])
have_survival <- isTRUE(pkg_status[["survival"]])

# Helper to safely library() optional packages if present
safe_library <- function(pkg) {
  if (requireNamespace(pkg, quietly = TRUE)) {
    suppressPackageStartupMessages(
      library(pkg, character.only = TRUE)
    )
    return(TRUE)
  }
  FALSE
}

invisible(lapply(optional_pkgs, safe_library))

cat("Setup complete.\n")


In [ ]:
# ============================================================
# SECTION 2: HELPER FUNCTIONS
# ============================================================

scale_rows <- function(mat) {
  mat <- as.matrix(mat)
  row_means <- rowMeans(mat, na.rm = TRUE)
  row_sds <- apply(mat, 1, sd, na.rm = TRUE)
  row_sds[row_sds == 0] <- 1
  out <- (mat - row_means) / row_sds
  return(out)
}

simulate_hidden_factor <- function(n, k = 1, sd = 1) {
  matrix(rnorm(n * k, mean = 0, sd = sd), nrow = n, ncol = k)
}

simulate_feature_loadings <- function(p, k = 1, sd = 1) {
  matrix(rnorm(p * k, mean = 0, sd = sd), nrow = p, ncol = k)
}

make_design <- function(group) {
  model.matrix(~ group)
}

make_null_design <- function(n) {
  model.matrix(~ 1, data = data.frame(x = rep(1, n)))
}

pca_surrogates <- function(expr, n_sv = 2) {
  expr <- as.matrix(expr)
  expr_centered <- t(scale(t(expr), center = TRUE, scale = FALSE))
  pc <- prcomp(t(expr_centered), center = FALSE, scale. = FALSE)
  sv <- pc$x[, seq_len(min(n_sv, ncol(pc$x))), drop = FALSE]
  colnames(sv) <- paste0("PC_SV_", seq_len(ncol(sv)))
  return(sv)
}

estimate_surrogates <- function(expr, mod, mod0, n_sv = NULL) {
  expr <- as.matrix(expr)
  n <- ncol(expr)

  if (!is.null(n_sv) && n_sv < 1) {
    stop("n_sv must be >= 1 if provided")
  }

  if (have_sva) {
    if (is.null(n_sv)) {
      n_sv <- tryCatch({
        sva::num.sv(expr, mod, method = "leek")
      }, error = function(e) {
        2
      })
      if (is.na(n_sv) || n_sv < 1) n_sv <- 2
    }
    svobj <- tryCatch({
      sva::sva(expr, mod = mod, mod0 = mod0, n.sv = n_sv)
    }, error = function(e) {
      NULL
    })
    if (!is.null(svobj) && !is.null(svobj$sv)) {
      sv <- svobj$sv
      colnames(sv) <- paste0("SV", seq_len(ncol(sv)))
      return(list(method = "sva", sv = sv))
    }
  }

  sv <- pca_surrogates(expr, n_sv = ifelse(is.null(n_sv), 2, n_sv))
  return(list(method = "pca_fallback", sv = sv))
}

fit_featurewise_lm <- function(expr, design, coef_index = 2) {
  expr <- as.matrix(expr)
  XtX_inv <- solve(crossprod(design))
  betas <- XtX_inv %*% t(design) %*% t(expr)
  fitted <- design %*% betas
  resid <- t(expr) - fitted
  df_resid <- nrow(design) - ncol(design)
  sigma2 <- colSums(resid^2) / df_resid
  se_beta <- sqrt(sigma2 * XtX_inv[coef_index, coef_index])
  t_stat <- betas[coef_index, ] / se_beta
  p_val <- 2 * pt(abs(t_stat), df = df_resid, lower.tail = FALSE)

  out <- data.frame(
    beta = as.numeric(betas[coef_index, ]),
    t = as.numeric(t_stat),
    p = as.numeric(p_val),
    stringsAsFactors = FALSE
  )
  out$padj <- p.adjust(out$p, method = "BH")
  return(out)
}

evaluate_hits <- function(res, truth) {
  stopifnot(length(truth) == nrow(res))
  detected <- res$padj < 0.05
  tp <- sum(detected & truth)
  fp <- sum(detected & !truth)
  fn <- sum(!detected & truth)
  tn <- sum(!detected & !truth)

  sensitivity <- if ((tp + fn) == 0) NA_real_ else tp / (tp + fn)
  specificity <- if ((tn + fp) == 0) NA_real_ else tn / (tn + fp)
  precision <- if ((tp + fp) == 0) NA_real_ else tp / (tp + fp)
  fdr_empirical <- if ((tp + fp) == 0) NA_real_ else fp / (tp + fp)

  data.frame(
    TP = tp,
    FP = fp,
    FN = fn,
    TN = tn,
    sensitivity = sensitivity,
    specificity = specificity,
    precision = precision,
    empirical_FDR = fdr_empirical
  )
}

plot_pvalue_hist <- function(p, main = "P-value histogram") {
  hist(
    p,
    breaks = 40,
    main = main,
    xlab = "p-value",
    border = "white"
  )
}

plot_sv_vs_covariate <- function(sv, covariate, main_prefix = "Association") {
  sv <- as.matrix(sv)
  oldpar <- par(no.readonly = TRUE)
  on.exit(par(oldpar))
  par(mfrow = c(1, ncol(sv)))
  for (i in seq_len(ncol(sv))) {
    plot(
      covariate, sv[, i],
      xlab = deparse(substitute(covariate)),
      ylab = colnames(sv)[i],
      main = paste(main_prefix, colnames(sv)[i])
    )
    abline(lm(sv[, i] ~ covariate), lty = 2)
  }
}

confusion_summary <- function(name, eval_df) {
  cat("----", name, "----\n")
  print(eval_df)
  cat("\n")
}

top_table <- function(res, ids = NULL, n = 10) {
  tt <- res[order(res$p, decreasing = FALSE), , drop = FALSE]
  if (!is.null(ids)) {
    tt$id <- ids
    tt <- tt[, c("id", setdiff(colnames(tt), "id"))]
  }
  head(tt, n)
}

correlate_sv_with_known <- function(sv, meta_df) {
  sv <- as.matrix(sv)
  out <- list()
  for (j in seq_len(ncol(sv))) {
    vals <- sapply(meta_df, function(x) {
      if (is.numeric(x)) {
        suppressWarnings(cor(sv[, j], x))
      } else {
        x_num <- as.numeric(as.factor(x))
        suppressWarnings(cor(sv[, j], x_num))
      }
    })
    out[[colnames(sv)[j]]] <- vals
  }
  do.call(rbind, out)
}


In [ ]:
# ============================================================
# SECTION 5: USE CASE 3
# DNA METHYLATION PROBE DATA WITH HIDDEN CHIP AND POSITION EFFECTS
# ============================================================

cat("Use case 3: Simulating methylation probe data...\n")

n_cpg <- 12000
n_samples3 <- 96

disease <- factor(rep(c("normal", "tumor"), each = n_samples3 / 2))
sex <- factor(sample(c("F", "M"), n_samples3, replace = TRUE))
age <- round(rnorm(n_samples3, mean = 58, sd = 9))

chip <- factor(rep(paste0("chip", 1:12), each = 8))
chip_position <- factor(rep(1:8, times = 12))

# Hidden technical and biological mixtures
cell_mix <- scale(rnorm(n_samples3) + ifelse(disease == "tumor", 0.8, -0.5))[, 1]
bisulfite_eff <- scale(rnorm(n_samples3) + as.numeric(chip) / 5)[, 1]
position_effect <- scale(rnorm(n_samples3) + as.numeric(chip_position) / 3)[, 1]

truth_dmp <- rep(FALSE, n_cpg)
truth_dmp[sample(seq_len(n_cpg), 900)] <- TRUE

baseline_m <- rnorm(n_cpg, mean = 0, sd = 0.7)
disease_effect_m <- rep(0, n_cpg)
disease_effect_m[truth_dmp] <- rnorm(sum(truth_dmp), mean = 0.7, sd = 0.2)

sex_effect_m <- rnorm(n_cpg, 0, 0.05)
age_effect_m <- rnorm(n_cpg, 0, 0.01)
cell_loading <- rnorm(n_cpg, 0, 0.8)
bisulfite_loading <- rnorm(n_cpg, 0, 0.9)
position_loading <- rnorm(n_cpg, 0, 0.6)

meth_mvals <- matrix(0, nrow = n_cpg, ncol = n_samples3)

for (i in seq_len(n_cpg)) {
  meth_mvals[i, ] <- baseline_m[i] +
    disease_effect_m[i] * (disease == "tumor") +
    sex_effect_m[i] * (sex == "M") +
    age_effect_m[i] * age +
    cell_loading[i] * cell_mix +
    bisulfite_loading[i] * bisulfite_eff +
    position_loading[i] * position_effect +
    rnorm(n_samples3, 0, 0.5)
}

rownames(meth_mvals) <- paste0("cg", sprintf("%08d", seq_len(n_cpg)))
colnames(meth_mvals) <- paste0("meth_sample_", seq_len(n_samples3))

# Convert M-values to pseudo-beta values for illustration
inv_logit <- function(x) 1 / (1 + exp(-x))
meth_beta <- inv_logit(meth_mvals)

meta_meth <- data.frame(
  sample = colnames(meth_mvals),
  disease = disease,
  sex = sex,
  age = age,
  chip = chip,
  chip_position = chip_position,
  cell_mix = cell_mix,
  bisulfite_eff = bisulfite_eff,
  position_effect = position_effect
)

cat("Methylation matrix dimensions:", dim(meth_mvals), "\n")
cat("True differential methylation probes:", sum(truth_dmp), "\n")

mod_meth_naive <- model.matrix(~ disease, data = meta_meth)
mod_meth_known <- model.matrix(~ disease + sex + age + chip, data = meta_meth)
mod0_meth_known <- model.matrix(~ sex + age + chip, data = meta_meth)

sv_meth <- estimate_surrogates(meth_mvals, mod = mod_meth_known, mod0 = mod0_meth_known, n_sv = 5)
design_meth_sva <- cbind(mod_meth_known, sv_meth$sv)

res_meth_naive <- fit_featurewise_lm(meth_mvals, mod_meth_naive, coef_index = 2)
res_meth_known <- fit_featurewise_lm(meth_mvals, mod_meth_known, coef_index = 2)
res_meth_sva <- fit_featurewise_lm(meth_mvals, design_meth_sva, coef_index = 2)

eval_meth_naive <- evaluate_hits(res_meth_naive, truth_dmp)
eval_meth_known <- evaluate_hits(res_meth_known, truth_dmp)
eval_meth_sva <- evaluate_hits(res_meth_sva, truth_dmp)

confusion_summary("Methylation naive", eval_meth_naive)
confusion_summary("Methylation known covariates", eval_meth_known)
confusion_summary("Methylation + SVA", eval_meth_sva)

cat("Top methylation probes after SVA:\n")
print(top_table(res_meth_sva, ids = rownames(meth_mvals), n = 12))

cat("Associations of methylation SVs with known technical/biological covariates:\n")
print(round(correlate_sv_with_known(sv_meth$sv, meta_meth), 3))

oldpar <- par(no.readonly = TRUE)
par(mfrow = c(2, 2))
plot_pvalue_hist(res_meth_naive$p, main = "Methylation naive")
plot_pvalue_hist(res_meth_known$p, main = "Methylation known")
plot_pvalue_hist(res_meth_sva$p, main = "Methylation + SVA")
plot(
  rowMeans(meth_beta[, disease == "normal", drop = FALSE]),
  rowMeans(meth_beta[, disease == "tumor", drop = FALSE]),
  pch = 16, cex = 0.3,
  xlab = "Mean beta: normal",
  ylab = "Mean beta: tumor",
  main = "Probe-wise beta value shift"
)
abline(0, 1, lty = 2)
par(oldpar)

# Use-case interpretation:
cat("\nInterpretation:\n")
cat("This synthetic methylation example mimics Illumina-style probe data where disease status,\n")
cat("chip-level artifacts, probe position effects, bisulfite conversion variability, and cell composition\n")
cat("can all introduce hidden structure. SVA attempts to recover latent factors that improve DMP detection.\n\n")
